<a href="https://colab.research.google.com/github/trainiacethan/Enigma-Machine/blob/main/Enigma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import collections
from collections import deque
# Enigma Machine - Ethan Pawar, 2026

def enigma():
  message = str(input("Enter your message: ")).upper() # uppercase
  alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
  rotor_order = []
  i = 0
  while i < 3:
    num = int(input("Enter rotor number (1-5), no repeats: "))
    if num in rotor_order:
      print("Invalid input. Try again!")
    elif num < 1 or num > 5:
      print("Invalid input. Try again!")
    else:
      rotor_order.append(num)
      i += 1
      print(str(rotor_order))


  rotors = {
    1: "EKMFLGDQVZNTOWYHXUSPAIBRCJ",
    2: "AJDKSIRUXBLHWTMCQGZNPYFVOE",
    3: "BDFHJLCPRTXVZNYEIWGAKMUSQO",
    4: "ESOVPZJAYQUIRHXLNFTGKDCMWB",
    5: "VZBRGITYUPSDNHLXAWMJQOFECK"
  }

  # Notches for each rotor (index of step causing letter)
  notches = {
      1: alphabet.find('Q'),
      2: alphabet.find('E'),
      3: alphabet.find('V'),
      4: alphabet.find('J'),
      5: alphabet.find('Z')
  }

###################################################################################################

# Reflector thingy
  reflector_b = "YRUHQSLDPXNGOKMIEBFZCWVJAT"

  # Initialize rotor offsets for the three chosen rotors
  rotor_offsets = [0, 0, 0]

# initial rotor settings from the user
  print("\nEnter initial rotor settings (A-Z) for the chosen rotors:")
  for i in range(3):
      while True:
          setting_char = input(f"Rotor {i+1} (Rotor {rotor_order[i]}): ").upper()
          if len(setting_char) == 1 and setting_char in alphabet:
              rotor_offsets[i] = alphabet.find(setting_char)
              break
          else:
              print("Invalid input. Please enter a single letter (A-Z).")
  print(f"Initial rotor offsets: {rotor_offsets}")

  def map_char_through_wiring(input_char, wiring, rotor_offset):
    if input_char not in alphabet:
        return input_char

    input_idx = alphabet.find(input_char)
    internal_input_idx = (input_idx + rotor_offset) % 26
    char_from_wiring = wiring[internal_input_idx]
    internal_output_idx = alphabet.find(char_from_wiring)
    final_output_idx = (internal_output_idx - rotor_offset + 26) % 26 # Add 26 to handle negative results of modulo correctly
    return alphabet[final_output_idx]

  def invert_wiring(wiring):
    inverted_map = [''] * len(alphabet)
    for i in range(len(alphabet)):
      original_char = alphabet[i]
      mapped_char = wiring[i]
      inverted_map[alphabet.find(mapped_char)] = original_char
    return "".join(inverted_map)

  encrypted_message = ""
  for letter in message:
    current_char = letter

    # Stepping logic
    # Rightmost rotor (rotor_order[2]) always steps
    # Middle rotor (rotor_order[1]) steps if its notch is reached OR if the rightmost rotor's notch is reached
    # Leftmost rotor (rotor_order[0]) steps if the middle rotor's notch is reached

    # Check for double stepping (when middle rotor is at its notch and steps, it also pushes the left rotor)
    if rotor_offsets[1] == notches[rotor_order[1]]:
        rotor_offsets[0] = (rotor_offsets[0] + 1) % 26 # Left rotor steps
        rotor_offsets[1] = (rotor_offsets[1] + 1) % 26 # Middle rotor steps
    # Check if rightmost rotor causes middle rotor to step
    elif rotor_offsets[2] == notches[rotor_order[2]]:
        rotor_offsets[1] = (rotor_offsets[1] + 1) % 26 # Middle rotor steps

    rotor_offsets[2] = (rotor_offsets[2] + 1) % 26 # Rightmost rotor always steps


    # forward Pass
    for i, rotor_num in enumerate(rotor_order):
      rotor_wiring = rotors[rotor_num]
      current_char = map_char_through_wiring(current_char, rotor_wiring, rotor_offsets[i])

    # Pass thru reflector
    current_char = map_char_through_wiring(current_char, reflector_b, 0)

    # Backward Pass
    for i, rotor_num in reversed(list(enumerate(rotor_order))):
      rotor_wiring = rotors[rotor_num]
      inverted_rotor_wiring = invert_wiring(rotor_wiring) # inverted wiring
      current_char = map_char_through_wiring(current_char, inverted_rotor_wiring, rotor_offsets[i])

    encrypted_message += current_char

  print("Original Message:", message)
  print("Encrypted Message:", encrypted_message)


###################################################################################################

enigma()